# S1 原始业务数据规则预处理

本 notebook 读取 `data/civ_point_info.csv`，仅处理 5 个必需字段，做规则型清洗、记录分类和样本元数据生成。

输出目录：`outputs/s1_preprocess/`

## Phase 1：加载与盘点

配置路径、常量与必需字段，读取 CSV 并校验。

In [1]:
from pathlib import Path
import json

import pandas as pd

# ---- 配置 ----
input_path = Path("data/civ_point_info.csv")
output_dir = Path("outputs/s1_preprocess")

sample_prefix = "ccmb"
image_extensions = {".jpg", ".jpeg", ".png"}
short_desc_threshold = 4  # prob_desc 字符数少于此值标记为待复核

required_input_columns = [
    "pk_id",
    "pt_type",
    "prob_desc",
    "prob_img",
    "insp_std",
]

sample_columns = [
    "sample_id",
    "source_pk_id",
    "image_path",
    "raw_primary_prob_img",
    "raw_prob_img",
    "local_image_path",
    "image_exists",
    "point_type",
    "inspection_standard_raw",
    "problem_description_raw",
    "include_flag",
    "exclude_reason",
    "review_status",
]

expected_output_files = {
    "sample_metadata.csv",
    "sample_metadata.jsonl",
    "candidate_records.csv",
    "excluded_records.csv",
    "pending_review_records.csv",
    "field_completeness.csv",
    "preprocess_summary.json",
}

# ---- 读取 ----
raw_df = pd.read_csv(input_path, low_memory=False)

missing = [c for c in required_input_columns if c not in raw_df.columns]
if missing:
    raise ValueError(f"原始表缺少必需字段：{missing}")

# 仅保留 5 个必需列，其余 11 列不读取不处理
raw_df = raw_df[required_input_columns].copy()

print(f"原始记录数：{len(raw_df)}")
print(f"字段（{len(raw_df.columns)}）：{list(raw_df.columns)}")
print(raw_df.dtypes)

原始记录数：44367
字段（5）：['pk_id', 'pt_type', 'prob_desc', 'prob_img', 'insp_std']
pk_id        int64
pt_type        str
prob_desc      str
prob_img       str
insp_std       str
dtype: object


## Phase 2：标准化

对 5 个必需列做统一清洗：去除首尾空白、空字符串转 `None`、图片扩展名小写化。

In [2]:
def _strip_and_nullify(series):
    """去除首尾空白，空字符串转为 None。"""
    def _clean(val):
        if val is None or pd.isna(val):
            return None
        text = str(val).strip()
        return None if text == "" else text
    return series.apply(_clean)


def _normalize_image_ext(path):
    """将图片路径中的大写扩展名转为小写（.JPG → .jpg 等）。"""
    if path is None or pd.isna(path):
        return path
    path_str = str(path)
    for ext in (".JPG", ".JPEG", ".PNG"):
        if path_str.endswith(ext):
            return path_str[:-len(ext)] + ext.lower()
    return path_str


cleaned_df = raw_df.copy()

# 所有字符串列：去空白 + 空字符串 → None
for col in cleaned_df.columns:
    if cleaned_df[col].dtype == object:
        cleaned_df[col] = _strip_and_nullify(cleaned_df[col])

# prob_img 扩展名小写化
cleaned_df["prob_img"] = cleaned_df["prob_img"].apply(_normalize_image_ext)

# 确认 pt_type 空格变体已修复
pt_type_before = raw_df["pt_type"].nunique()
pt_type_after = cleaned_df["pt_type"].nunique()
if pt_type_before != pt_type_after:
    print(f"pt_type 唯一值：{pt_type_before} → {pt_type_after}（修复了空格变体）")

print("标准化完成。")
print(cleaned_df.head().to_string(index=False))

标准化完成。
 pk_id pt_type        prob_desc                                                                               prob_img insp_std
   757    居民小区        南门处地面存在垃圾 /profile/upload/2024/04/16/tmp_6c91aef161b7b1e2029bc11c5744ad27_20240416084310A203.jpg     公共环境
   758    居民小区  小区南门处公益广告存在老旧褪色 /profile/upload/2024/04/16/tmp_c1a803b95072ca015a35df44727c7196_20240416084418A204.jpg     公益宣传
   759    居民小区      3栋2单元一楼存在烟头 /profile/upload/2024/04/16/tmp_2cfabd46c75b6661bda0a30ae7c3c6e7_20240416084702A205.jpg     公共环境
   760    居民小区 3栋2单元二层半窗户台上存在垃圾 /profile/upload/2024/04/16/tmp_a41c7999c1c83f9d13a3f65e13da7e35_20240416084812A207.jpg     公益宣传
   761    居民小区    3栋2单元三楼地面存在垃圾 /profile/upload/2024/04/16/tmp_5476886260826e84022305609bc096b8_20240416084912A208.jpg     公共环境


## Phase 3：质量画像

统计逐列完整性、prob_desc 长度分布和 pt_type/insp_std 高频值。

In [3]:
total_records = len(raw_df)

field_completeness_rows = []
for col in required_input_columns:
    missing_count = int(raw_df[col].isna().sum())
    empty_or_invalid_count = int(cleaned_df[col].isna().sum())
    non_empty_count = total_records - empty_or_invalid_count
    field_completeness_rows.append({
        "field": col,
        "total_records": total_records,
        "missing_count": missing_count,
        "empty_or_invalid_count": empty_or_invalid_count,
        "non_empty_count": non_empty_count,
        "non_empty_ratio": round(non_empty_count / total_records, 6),
    })

field_completeness = pd.DataFrame(field_completeness_rows)
print(field_completeness.sort_values("empty_or_invalid_count", ascending=False).to_string(index=False))

# prob_desc 长度分布
desc_lengths = cleaned_df["prob_desc"].dropna().str.len()
print(f"\nprob_desc 长度分布：min={desc_lengths.min()}, median={desc_lengths.median():.0f}, max={desc_lengths.max()}, mean={desc_lengths.mean():.1f}")

# pt_type / insp_std 高频值
print("\npt_type Top 10:")
print(cleaned_df["pt_type"].value_counts().head(10).to_string())
print("\ninsp_std Top 10:")
print(cleaned_df["insp_std"].value_counts().head(10).to_string())

    field  total_records  missing_count  empty_or_invalid_count  non_empty_count  non_empty_ratio
 insp_std          44367           1255                    1255            43112         0.971713
 prob_img          44367            276                     276            44091         0.993779
prob_desc          44367              7                       7            44360         0.999842
    pk_id          44367              0                       0            44367         1.000000
  pt_type          44367              0                       0            44367         1.000000

prob_desc 长度分布：min=2, median=10, max=63, mean=10.4

pt_type Top 10:
pt_type
居民小区           14295
乡镇              9331
主次干道/商业大街       6407
农贸(集贸) 市场       2555
行政村             1917
行政村新时代文明实践站     1289
乡镇新时代文明实践所      1196
背街小巷            1186
中小学校            1010
 大型商超            694

insp_std Top 10:
insp_std
公共秩序    18473
公共环境    16022
基础设施     3047
服务展示     2613
公益宣传     1558
文明行为      861
宣传展示      389


## Phase 4：记录分类

每条记录按规则分为候选 / 排除 / 待复核：

- **排除**：5 个必需字段任一缺失，或 prob_img 含多图（逗号分隔）
- **待复核**：prob_desc 字符数不足（< 4 字），不排除仅标记

In [4]:
# ---- 构建排除原因（每条记录收集全部原因，非仅第一个）----
n = len(cleaned_df)
exclude_reasons = pd.Series([[] for _ in range(n)], index=cleaned_df.index, dtype=object)

# 1. 缺失字段检查
for col in required_input_columns:
    for idx in cleaned_df.index[cleaned_df[col].isna()]:
        exclude_reasons.at[idx].append(f"missing_{col}")

# 2. 多图检查（仅对有 prob_img 的记录）
has_img = cleaned_df["prob_img"].notna()
for idx in has_img[has_img].index:
    prob_images = [p.strip() for p in str(cleaned_df.at[idx, "prob_img"]).split(",") if p.strip()]
    if len(prob_images) > 1:
        exclude_reasons.at[idx].append("multiple_prob_images")

# ---- 候选 / 排除分流 ----
is_excluded = exclude_reasons.apply(len) > 0
is_candidate = ~is_excluded

candidate_records = cleaned_df.loc[is_candidate].copy().reset_index(drop=True)
excluded_records = cleaned_df.loc[is_excluded].copy().reset_index(drop=True)
excluded_records["exclude_reason"] = exclude_reasons[is_excluded].apply("; ".join).values

# ---- 待复核标记（仅对候选记录）----
review_reasons = pd.Series([[] for _ in range(n)], index=cleaned_df.index, dtype=object)
short_mask = is_candidate & cleaned_df["prob_desc"].str.len().lt(short_desc_threshold)
for idx in short_mask[short_mask].index:
    review_reasons.at[idx].append("short_description")

# 构建 pending_review_records
pending_rows = []
for idx in review_reasons.index[review_reasons.apply(len) > 0]:
    row = cleaned_df.loc[idx]
    for reason in review_reasons.at[idx]:
        pending_rows.append({
            "pk_id": row["pk_id"],
            "pt_type": row["pt_type"],
            "prob_desc": row["prob_desc"],
            "prob_img": row["prob_img"],
            "insp_std": row["insp_std"],
            "review_reason": reason,
        })

pending_review_records = pd.DataFrame(
    pending_rows,
    columns=required_input_columns + ["review_reason"],
)

print(f"候选记录数：{len(candidate_records)}")
print(f"排除记录数：{len(excluded_records)}")
if len(excluded_records) > 0:
    print(excluded_records["exclude_reason"].value_counts().to_string())
print(f"\n待复核记录数：{len(pending_review_records)}")
if len(pending_review_records) > 0:
    print(pending_review_records["review_reason"].value_counts().to_string())

候选记录数：39039
排除记录数：5328
exclude_reason
multiple_prob_images                      3802
missing_insp_std                          1178
missing_prob_img                           267
missing_insp_std; multiple_prob_images      65
missing_prob_img; missing_insp_std           9
missing_prob_desc                            4
missing_prob_desc; missing_insp_std          3

待复核记录数：435
review_reason
short_description    435


## Phase 5：图片处理

替换图片路径前缀（/profile/upload → 本地挂载路径），校验每个样本的图片文件是否存在。

In [5]:
# 路径前缀映射：CSV 中 /profile/upload → 本地挂载路径
RAW_PREFIX = "/profile/upload"
LOCAL_PREFIX = "/mnt/d/BaiduNetdiskDownload/ruoyi/uploadPath/upload"

def resolve_local_path(raw_path):
    """将 CSV 中的 /profile/upload 前缀替换为本地挂载路径"""
    if raw_path is None or pd.isna(raw_path):
        return None
    path_str = str(raw_path)
    if path_str.startswith(RAW_PREFIX):
        return path_str.replace(RAW_PREFIX, LOCAL_PREFIX, 1)
    return path_str

# 为候选记录生成本地路径
candidate_records["local_image_path"] = candidate_records["prob_img"].apply(resolve_local_path)

# 校验图片是否存在
image_exists = []
missing_count = 0
for p in candidate_records["local_image_path"]:
    if p is None:
        image_exists.append(False)
        missing_count += 1
    else:
        exists = Path(p).exists()
        image_exists.append(exists)
        if not exists:
            missing_count += 1

candidate_records["image_exists"] = image_exists

multi_image_records = 0  # 已在 Phase 4 排除

print(f"路径前缀已替换: {RAW_PREFIX} → {LOCAL_PREFIX}")
print(f"可用图片: {sum(image_exists)}/{len(candidate_records)}")
print(f"缺失图片: {missing_count}")
if missing_count > 0:
    print(f"缺失率: {missing_count/len(candidate_records)*100:.2f}%")

路径前缀已替换: /profile/upload → /mnt/d/BaiduNetdiskDownload/ruoyi/uploadPath/upload
可用图片: 39039/39039
缺失图片: 0


## Phase 6：样本元数据

为每条候选记录生成 `sample_id`，组装完整的样本元数据。

In [6]:
sample_rows = []

for sample_index, (_, row) in enumerate(candidate_records.iterrows(), start=1):
    sample_id = f"{sample_prefix}_{sample_index:06d}"
    raw_prob_img = row["prob_img"]

    # 多图已在 Phase 5 排除，直接取 prob_img
    raw_primary_prob_img = raw_prob_img

    suffix = Path(str(raw_primary_prob_img) if raw_primary_prob_img is not None else "").suffix.lower()
    if suffix not in image_extensions:
        suffix = ".jpg"
    image_path = f"images/raw/{sample_id}{suffix}"

    sample_rows.append({
        "sample_id": sample_id,
        "source_pk_id": row["pk_id"],
        "image_path": image_path,
        "raw_primary_prob_img": raw_primary_prob_img,
        "raw_prob_img": raw_prob_img,
        "local_image_path": row["local_image_path"],
        "image_exists": row["image_exists"],
        "point_type": row["pt_type"],
        "inspection_standard_raw": row["insp_std"],
        "problem_description_raw": row["prob_desc"],
        "include_flag": True,
        "exclude_reason": None,
        "review_status": "pending",
    })

sample_metadata = pd.DataFrame(sample_rows, columns=sample_columns)

print(f"样本元数据记录数：{len(sample_metadata)}")
print(sample_metadata.head().to_string(index=False))

样本元数据记录数：39039
  sample_id  source_pk_id                 image_path                                                                   raw_primary_prob_img                                                                           raw_prob_img                                                                                                           local_image_path  image_exists point_type inspection_standard_raw problem_description_raw  include_flag exclude_reason review_status
ccmb_000001           757 images/raw/ccmb_000001.jpg /profile/upload/2024/04/16/tmp_6c91aef161b7b1e2029bc11c5744ad27_20240416084310A203.jpg /profile/upload/2024/04/16/tmp_6c91aef161b7b1e2029bc11c5744ad27_20240416084310A203.jpg /mnt/d/BaiduNetdiskDownload/ruoyi/uploadPath/upload/2024/04/16/tmp_6c91aef161b7b1e2029bc11c5744ad27_20240416084310A203.jpg          True       居民小区                    公共环境               南门处地面存在垃圾          True           None       pending
ccmb_000002           758 images/raw/ccmb_000002.jpg 

## Phase 7：汇总与输出

生成预处理摘要，写出全部 7 个输出文件。

In [7]:
# ---- 汇总 ----
missing_counts = {}
for col in required_input_columns:
    missing_counts[col] = int(cleaned_df[col].isna().sum())

top_point_types = {
    str(k): int(v)
    for k, v in cleaned_df["pt_type"].fillna("<NA>").value_counts().head(20).items()
}
top_inspection_standards = {
    str(k): int(v)
    for k, v in cleaned_df["insp_std"].fillna("<NA>").value_counts().head(20).items()
}

available_images = int(candidate_records["image_exists"].sum())

summary = {
    "total_records": int(len(cleaned_df)),
    "candidate_records": int(len(candidate_records)),
    "excluded_records": int(len(excluded_records)),
    "pending_review_records": int(len(pending_review_records)),
    "multi_image_records": int(multi_image_records),
    "missing_local_images": int(missing_count),
    "image_availability_ratio": round(available_images / len(candidate_records), 6),
    "missing_counts": missing_counts,
    "top_point_types": top_point_types,
    "top_inspection_standards": top_inspection_standards,
    "input_path": str(input_path),
    "output_dir": str(output_dir),
}

# ---- 写出 ----
output_dir.mkdir(parents=True, exist_ok=True)

sample_metadata.to_csv(output_dir / "sample_metadata.csv", index=False, encoding="utf-8-sig")
candidate_records.to_csv(output_dir / "candidate_records.csv", index=False, encoding="utf-8-sig")
excluded_records.to_csv(output_dir / "excluded_records.csv", index=False, encoding="utf-8-sig")
pending_review_records.to_csv(output_dir / "pending_review_records.csv", index=False, encoding="utf-8-sig")
field_completeness.to_csv(output_dir / "field_completeness.csv", index=False, encoding="utf-8-sig")

with (output_dir / "sample_metadata.jsonl").open("w", encoding="utf-8") as f:
    for record in sample_metadata.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

with (output_dir / "preprocess_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("预处理输出文件：")
for name in sorted(p.name for p in output_dir.iterdir()):
    print(f"  {name}")

print(json.dumps({
    "total_records": summary["total_records"],
    "candidate_records": summary["candidate_records"],
    "excluded_records": summary["excluded_records"],
    "pending_review_records": summary["pending_review_records"],
    "missing_local_images": summary["missing_local_images"],
    "image_availability_ratio": summary["image_availability_ratio"],
}, ensure_ascii=False, indent=2))

预处理输出文件：
  candidate_records.csv
  excluded_records.csv
  field_completeness.csv
  pending_review_records.csv
  preprocess_summary.json
  sample_metadata.csv
  sample_metadata.jsonl
{
  "total_records": 44367,
  "candidate_records": 39039,
  "excluded_records": 5328,
  "pending_review_records": 435,
  "missing_local_images": 0,
  "image_availability_ratio": 1.0
}


## Phase 8：构建 Tiny 数据集

从候选记录中通过 `pt_type` 分层抽样构建 5 条小样本数据集，用于快速验证与调试。

- **抽样策略**：按 `pt_type` 分层抽样（每类至少 1 条），确保点位类型多样性
- **过滤条件**：仅选取 `image_exists=True` 的记录
- **随机种子**：`random_state=42`
- **输出目录**：`data/tiny/`
  - `metadata.csv` — 5 条元数据
  - `images/` — 拷贝的图片文件，命名为 `{sample_id}.jpg`
  - `summary.json` — 数据集统计信息

In [8]:
import shutil
from pathlib import Path
import json

import pandas as pd

# ---- 配置 ----
candidate_path = Path("outputs/s1_preprocess/candidate_records.csv")
metadata_path = Path("outputs/s1_preprocess/sample_metadata.csv")
output_dir = Path("data/tiny")
images_dir = output_dir / "images"

RANDOM_STATE = 42
TARGET_SIZE = 5

# ---- 读取 ----
candidate_df = pd.read_csv(candidate_path, encoding="utf-8-sig")
metadata_df = pd.read_csv(metadata_path, encoding="utf-8-sig")

# 仅保留 image_exists=True 的记录
candidate_df = candidate_df[candidate_df["image_exists"] == True].copy()
valid_pks = set(candidate_df["pk_id"])
metadata_df = metadata_df[metadata_df["source_pk_id"].isin(valid_pks)].copy()

print(f"可用记录数: {len(candidate_df)}")
print(f"pt_type 类别数: {candidate_df['pt_type'].nunique()}")

# ---- 分层抽样 ----
pt_type_counts = candidate_df["pt_type"].value_counts()
total = len(candidate_df)

# 为每个 pt_type 计算分配名额（至少 1，不超过该类实际数量）
allocated = {}
for pt, count in pt_type_counts.items():
    n = max(1, round(count / total * TARGET_SIZE))
    allocated[pt] = min(n, count)

print(f"初步分配: {sum(allocated.values())} 条（{len(allocated)} 类）")

# 执行抽样
sampled_rows = []
for pt, n_sample in allocated.items():
    group = candidate_df[candidate_df["pt_type"] == pt]
    sampled_rows.append(group.sample(n=n_sample, random_state=RANDOM_STATE))

sampled_df = pd.concat(sampled_rows, ignore_index=True)

# 修正四舍五入导致的总数偏差
if len(sampled_df) > TARGET_SIZE:
    sampled_df = sampled_df.sample(n=TARGET_SIZE, random_state=RANDOM_STATE)
elif len(sampled_df) < TARGET_SIZE:
    existing_pks = set(sampled_df["pk_id"])
    remaining = candidate_df[~candidate_df["pk_id"].isin(existing_pks)]
    extra = remaining.sample(n=TARGET_SIZE - len(sampled_df), random_state=RANDOM_STATE)
    sampled_df = pd.concat([sampled_df, extra], ignore_index=True)

print(f"实际抽样: {len(sampled_df)} 条，覆盖 {sampled_df['pt_type'].nunique()} 种 pt_type")

# ---- 组装 metadata ----
sampled_pks = set(sampled_df["pk_id"])
sampled_meta = metadata_df[metadata_df["source_pk_id"].isin(sampled_pks)].copy()
pk_to_sample_id = dict(zip(sampled_meta["source_pk_id"], sampled_meta["sample_id"]))

meta_out = sampled_meta[[
    "sample_id", "image_path", "point_type",
    "inspection_standard_raw", "problem_description_raw"
]].copy()
meta_out = meta_out.rename(columns={
    "inspection_standard_raw": "inspection_standard",
    "problem_description_raw": "problem_description",
})
# 更新 image_path 为 tiny 内部相对路径
meta_out["image_path"] = meta_out["sample_id"].apply(lambda sid: f"images/{sid}.jpg")

# ---- 创建输出目录 ----
output_dir.mkdir(parents=True, exist_ok=True)
images_dir.mkdir(parents=True, exist_ok=True)

meta_out.to_csv(output_dir / "metadata.csv", index=False, encoding="utf-8-sig")
print(f"metadata.csv: {len(meta_out)} 条")

# ---- 拷贝图片 ----
copied = 0
missing = 0
for _, row in sampled_df.iterrows():
    src = Path(row["local_image_path"])
    sample_id = pk_to_sample_id.get(row["pk_id"])
    if sample_id is None:
        continue
    if not src.exists():
        missing += 1
        print(f"  警告: 图片缺失 pk_id={row['pk_id']}")
        continue
    suffix = src.suffix.lower()
    if suffix not in {".jpg", ".jpeg", ".png"}:
        suffix = ".jpg"
    dst = images_dir / f"{sample_id}{suffix}"
    shutil.copy2(src, dst)
    copied += 1

print(f"图片拷贝: {copied} 成功, {missing} 缺失")

# ---- summary.json ----
pt_dist = meta_out["point_type"].value_counts().to_dict()
pt_dist_out = {str(k): int(v) for k, v in pt_dist.items()}
insp_dist = sampled_meta["inspection_standard_raw"].value_counts().to_dict()
insp_dist_out = {str(k): int(v) for k, v in insp_dist.items()}

summary = {
    "dataset_name": "tiny",
    "description": f"从 candidate_records.csv 按 pt_type 分层抽样构建的 {TARGET_SIZE} 条小样本数据集",
    "source": "outputs/s1_preprocess/",
    "random_state": RANDOM_STATE,
    "total_records": int(len(meta_out)),
    "unique_point_types": int(meta_out["point_type"].nunique()),
    "images_copied": copied,
    "images_missing": missing,
    "point_type_distribution": pt_dist_out,
    "inspection_standard_distribution": insp_dist_out,
}

with (output_dir / "summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# ---- 最终报告 ----
print()
print("=" * 50)
print(f"Tiny 数据集生成完成 ({TARGET_SIZE} 条)")
print("=" * 50)
print(f"  输出目录:     {output_dir}")
print(f"  metadata.csv: {len(meta_out)} 条")
print(f"  images/:      {copied} 个文件")
print(f"  summary.json: 已生成")
print(f"  pt_type 覆盖: {meta_out['point_type'].nunique()} / {candidate_df['pt_type'].nunique()} 种")

可用记录数: 39039
pt_type 类别数: 50
初步分配: 51 条（50 类）
实际抽样: 5 条，覆盖 5 种 pt_type
metadata.csv: 5 条
图片拷贝: 5 成功, 0 缺失

Tiny 数据集生成完成 (5 条)
  输出目录:     data/tiny
  metadata.csv: 5 条
  images/:      5 个文件
  summary.json: 已生成
  pt_type 覆盖: 5 / 50 种


## Phase 9：导出 data-juicer 兼容 JSONL

从 Phase 8 生成的 tiny 数据集 metadata 导出 data-juicer 格式的 `tiny.jsonl`，包含 `<__dj__image>` 特殊标记用于多模态 MLLM 推理。

In [9]:
# ---- 配置 ----
tiny_dir = Path("data/tiny")
metadata_csv = tiny_dir / "metadata.csv"
output_jsonl = tiny_dir / "tiny.jsonl"

# ---- 读取 Phase 8 生成的 metadata ----
meta_df = pd.read_csv(metadata_csv, encoding="utf-8-sig")

# ---- 组装 data-juicer JSONL 格式 ----
# 格式: {"text": "<__dj__image>问题描述", "images": ["images/xxx.jpg"], "meta": {...}}
records = []
for _, row in meta_df.iterrows():
    records.append({
        "text": f"<__dj__image>{row['problem_description']}",
        "images": [row["image_path"]],
        "meta": {
            "sample_id": row["sample_id"],
            "point_type": row["point_type"],
            "inspection_standard": row["inspection_standard"],
        },
    })

# ---- 写出 JSONL ----
with output_jsonl.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"tiny.jsonl 导出完成: {output_jsonl} ({len(records)} 条)")
print(f"\n格式示例:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

tiny.jsonl 导出完成: data/tiny/tiny.jsonl (5 条)

格式示例:
{
  "text": "<__dj__image>城乡候车大厅门前树穴旁边有烟头",
  "images": [
    "images/ccmb_000019.jpg"
  ],
  "meta": {
    "sample_id": "ccmb_000019",
    "point_type": "交通场站周边",
    "inspection_standard": "公共环境"
  }
}


## Phase 10：生成 MLLM 富集提示词 JSONL

从 Phase 9 生成的 `tiny.jsonl` 构建 MLLM 富集提示词 JSONL（`tiny_enriched_prompt.jsonl`），供 `urban_incivility_enriched_Qwen2-VL-2B_7GB.yaml` 使用。

- **去除** `<__dj__image>` 内部标记（MLLM 通过 `images` 字段单独接收图片，直传 token 会干扰模型）
- **包装** 问题描述为结构化分析提示词，要求模型判断并描述不文明现象

In [ ]:
# ---- 配置 ----
tiny_jsonl = Path("data/tiny/tiny.jsonl")
output_jsonl = Path("data/tiny/tiny_enriched_prompt.jsonl")

# ---- MLLM 提示词模板 ----
PROMPT_TEMPLATE = (
    "请仔细观察这张城市管理图片，判断图片中是否存在以下不文明现象：{problem_description}。"
    "如果存在，请详细描述该不文明现象的具体表现，包括位置、严重程度等信息；"
    "如果不存在，请说明图片的实际内容。"
)

# ---- 读取 tiny.jsonl 并生成富集提示词 ----
records = []
with tiny_jsonl.open("r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line.strip())
        # 去除 <__dj__image> 内部标记，构建 MLLM 可直接理解的 prompt
        clean_desc = rec["text"].replace("<__dj__image>", "")
        enriched_prompt = PROMPT_TEMPLATE.format(problem_description=clean_desc)
        records.append({
            "text": enriched_prompt,
            "images": rec["images"],
            "meta": rec.get("meta", {}),
        })

# ---- 写出 ----
with output_jsonl.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"tiny_enriched_prompt.jsonl 生成完成: {output_jsonl} ({len(records)} 条)")
print(f"\n第一条记录预览:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))